In [0]:
# ============================================================
# 00.1 BIBLIOTECAS DO FRAMEWORK
# ============================================================

# O QUE FAZ:
# Centraliza as bibliotecas utilizadas pelos notebooks do Data Quality Framework.

# COMO FAZ:
# Importa as funções e classes PySpark utilizadas pelos Profiles,Data Preparation e demais componentes do framework.

# POR QUE É IMPORTANTE:
# Evita a duplicação de imports em diferentes notebooks e estabelece um ponto único de manutenção das dependências técnicas do framework.

# PERGUNTA RESPONDIDA:
# "Quais bibliotecas são utilizadas pelo framework?"

from pyspark.sql import functions as F

from itertools import combinations

from pyspark.sql.types import (
    NumericType,
    StringType,
    DateType,
    TimestampType,
    BooleanType,
    MapType,
    StructType,
    FloatType,
    DoubleType,
    ArrayType
)

from pyspark.sql import Window

In [0]:
# ============================================================
# FUNÇÃO - NORMALIZAÇÃO DE VALORES
# ============================================================

# O QUE FAZ:
# Converte os valores das colunas para uma representação textual adequada à análise de padrões.

# COMO FAZ:
# Converte o valor para string e remove espaços nas extremidades.

# POR QUE É IMPORTANTE:
# Permite aplicar as mesmas regras de Pattern Profile sobre diferentes tipos de dados.

# PERGUNTA RESPONDIDA:
# "Como representar diferentes tipos de valores em uma estrutura comum para análise de padrões?"

def normalize_pattern_value(column):
    return F.trim(column.cast("string"))

In [0]:
# ============================================================
# FUNÇÃO - COMPRIMENTO DO VALOR
# ============================================================

# O QUE FAZ:
# Calcula o comprimento dos valores utilizados na análise de padrões.

# COMO FAZ:
# Converte o valor para string, remove espaços externos e calcula a quantidade de caracteres.

# POR QUE É IMPORTANTE:
# O comprimento é uma das primeiras características estruturais utilizadas para identificar padrões e possíveis desvios.

# PERGUNTA RESPONDIDA:
# "Qual é o comprimento dos valores presentes na coluna?"

def add_pattern_length(column):
    valor = normalize_pattern_value(column)

    return F.length(valor)

In [0]:
# ============================================================
# FUNÇÃO - COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Classifica cada valor de acordo com sua composição de caracteres.

# COMO FAZ:
# Utiliza expressões regulares para identificar valores:
#
# NUMÉRICOS
# ALFABÉTICOS
# ALFANUMÉRICOS
# SÍMBOLOS
# MISTOS

# POR QUE É IMPORTANTE:
# Permite identificar alterações estruturais mesmo quando o valor possui o mesmo significado aparente.

# PERGUNTA RESPONDIDA:
# "Qual é a composição estrutural dos valores da coluna?"

def classify_character_composition(column):

    valor = normalize_pattern_value(column)

    return (
        F.when(
            valor.isNull() | (valor == ""),
            "VAZIO"
        )
        .when(
            valor.rlike(r"^[0-9]+$"),
            "NUMÉRICO"
        )
        .when(
            valor.rlike(r"^[A-Za-zÀ-ÿ]+$"),
            "ALFABÉTICO"
        )
        .when(
            valor.rlike(r"^[A-Za-zÀ-ÿ0-9]+$") &
            valor.rlike(r"[A-Za-zÀ-ÿ]") &
            valor.rlike(r"[0-9]"),
            "ALFANUMÉRICO"
        )
        .when(
            valor.rlike(r"^[^A-Za-zÀ-ÿ0-9]+$"),
            "SÍMBOLOS"
        )
        .otherwise(
            "MISTO"
        )
    )

In [0]:
# ============================================================
# FUNÇÃO - ESTRUTURA DO VALOR
# ============================================================

# O QUE FAZ:
# Converte cada caractere do valor em uma representação estrutural padronizada.

# COMO FAZ:
# Classifica:
#
# Letras  -> L
# Números -> N
# Espaços -> S
# Outros  -> X
#
# Exemplo:
#
# ABC123      -> LLLNNN
# 123456      -> NNNNNN
# AB-123      -> LLXNNN
# 12/05/2026  -> NN X NN X NNNN

# POR QUE É IMPORTANTE:
# Permite identificar padrões estruturais sem depender do significado semântico ou do nome da coluna.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura dos valores presentes na coluna?"

def generate_pattern_structure(column):

    valor = normalize_pattern_value(column)

    return (
        F.when(
            valor.isNull() | (valor == ""),
            "VAZIO"
        )
        .otherwise(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(
                            valor,
                            r"[A-Za-zÀ-ÿ]",
                            "L"
                        ),
                        r"[0-9]",
                        "N"
                    ),
                    r"\s",
                    "S"
                ),
                r"[^A-Za-zÀ-ÿ0-9\s]",
                "X"
            )
        )
    )

In [0]:
# ============================================================
# FUNÇÃO - RESUMO DOS PADRÕES
# ============================================================

# O QUE FAZ:
# Calcula a estrutura predominante de cada coluna e sua respectiva participação percentual.

# COMO FAZ:
# Conta a frequência de cada estrutura encontrada e utiliza uma janela para identificar a estrutura mais frequente.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente qual estrutura domina uma determinada coluna.

# PERGUNTA RESPONDIDA:
# "Qual é o padrão estrutural predominante de cada coluna?"

def generate_pattern_summary(df, columns, total_records):

    resultados = []

    for coluna in columns:

        estrutura_coluna = (
            df
            .select(
                F.lit(coluna).alias("coluna"),
                generate_pattern_structure(
                    F.col(coluna)
                ).alias("estrutura")
            )
            .groupBy(
                "coluna",
                "estrutura"
            )
            .agg(
                F.count("*").alias("frequencia")
            )
            .withColumn(
                "pct_estrutura",
                F.col("frequencia")
                / F.lit(total_records)
                * 100
            )
        )

        resultados.append(estrutura_coluna)

    resultado = resultados[0]

    for df_resultado in resultados[1:]:
        resultado = resultado.unionByName(df_resultado)

    window_pattern = (
        Window
        .partitionBy("coluna")
        .orderBy(
            F.desc("frequencia"),
            F.asc("estrutura")
        )
    )

    return (
        resultado
        .withColumn(
            "ranking",
            F.row_number().over(window_pattern)
        )
    )

In [0]:
# ============================================================
# FUNÇÃO - RESUMO DA COMPOSIÇÃO DE CARACTERES
# ============================================================

# O QUE FAZ:
# Calcula a distribuição das categorias de composição dos valores em cada coluna.

# COMO FAZ:
# Classifica cada valor e calcula frequência e percentual de cada categoria estrutural.

# POR QUE É IMPORTANTE:
# Permite identificar mudanças na composição dos dados, como mistura inesperada entre números, letras e símbolos.

# PERGUNTA RESPONDIDA:
# "Qual composição de caracteres predomina em cada coluna?"

def generate_character_composition(df, columns, total_records):

    resultados = []

    for coluna in columns:

        composicao = (
            df
            .select(
                F.lit(coluna).alias("coluna"),
                classify_character_composition(
                    F.col(coluna)
                ).alias("composicao")
            )
            .groupBy(
                "coluna",
                "composicao"
            )
            .agg(
                F.count("*").alias("frequencia")
            )
            .withColumn(
                "pct_composicao",
                F.col("frequencia")
                / F.lit(total_records)
                * 100
            )
        )

        resultados.append(composicao)

    resultado = resultados[0]

    for df_resultado in resultados[1:]:
        resultado = resultado.unionByName(df_resultado)

    return resultado

In [0]:
# ============================================================
# FUNÇÃO - GENÉRICAS DE DUPLICITY PROFILE
# ============================================================

# O QUE FAZ: referencia nomes de colunas com caracteres especiais.

# COMO FAZ: escapa crases e usa identificador delimitado do Spark.

# POR QUE IMPORTANTE: evita interpretação de pontos como acesso a campos aninhados.

def dq_col(nome):
    return F.col("`" + nome.replace("`", "``") + "`")

# O QUE FAZ: define se um valor é elegível para análise de unicidade.

# COMO FAZ: descarta NULL, NaN e opcionalmente strings vazias/whitespace.

# POR QUE IMPORTANTE: padroniza a política de ausência entre análises.

def dq_valid_value(coluna, tipo, ignorar_texto_vazio=True):
    c = dq_col(coluna)
    valido = c.isNotNull()
    if isinstance(tipo, (FloatType, DoubleType)):
        valido = valido & (~F.isnan(c))
    if ignorar_texto_vazio and isinstance(tipo, StringType):
        valido = valido & (F.length(F.trim(c)) > 0)
    return valido

# O QUE FAZ: indica elegibilidade de tipos para igualdade e countDistinct.

# COMO FAZ: exclui tipos complexos até existir política explícita de normalização.

# POR QUE IMPORTANTE: impede conclusões incorretas em dados não atômicos.

def dq_is_atomic(tipo):
    return not isinstance(tipo, (ArrayType, MapType, StructType))

# O QUE FAZ: calcula métricas de grupos duplicados de um DataFrame agregado.

# COMO FAZ: utiliza contagens por grupo e soma excedentes / envolvidos.

# POR QUE IMPORTANTE: diferencia registros duplicados de registros excedentes.

def dq_duplicate_group_metrics(grupos_df, coluna_frequencia="ocorrencias"):
    f = F.col(coluna_frequencia)
    return grupos_df.agg(
        F.count(F.lit(1)).alias("grupos_distintos"),
        F.sum(F.when(f > 1, 1).otherwise(0)).alias("grupos_duplicados"),
        F.sum(F.when(f > 1, f).otherwise(0)).alias("registros_em_grupos_duplicados"),
        F.sum(f - 1).alias("linhas_duplicadas_excedentes"),
        F.max(f).alias("maior_grupo")
    )

In [0]:

# ============================================================
# FUNÇÃO - MÉTRICAS DE DEPENDÊNCIA FUNCIONAL
# ============================================================

# O QUE FAZ:
# Consolida indicadores de conflitos em dependências
# funcionais candidatas.

# COMO FAZ:
# Agrega grupos previamente calculados, identificando
# determinantes associados a múltiplos valores dependentes.

# POR QUE É IMPORTANTE:
# Padroniza o cálculo das métricas relacionais e permite
# sua reutilização em diferentes Profiles.

# PERGUNTA RESPONDIDA:
# "Quantos determinantes apresentam valores conflitantes?"

def dq_consistency_fd_aggregate(grupos_df):

    conflito = (
        F.col("valores_dependentes_distintos") > 1
    )

    return grupos_df.agg(
        F.sum("registros").alias("registros_validos"),

        F.count(F.lit(1)).alias(
            "determinantes_distintos"
        ),

        F.sum(
            F.when(conflito, 1).otherwise(0)
        ).alias("determinantes_conflitantes"),

        F.sum(
            F.when(
                conflito,
                F.col("registros")
            ).otherwise(0)
        ).alias("registros_em_grupos_conflitantes")
    )


In [0]:

# ============================================================
# FUNÇÃO - GENÉRICAS DE OUTLIER PROFILE
# ============================================================

# O QUE FAZ:
# Padroniza referências seguras de colunas, limites e
# indicadores estatísticos de outliers.
#
# COMO FAZ:
# Utiliza expressões Spark e cálculos matemáticos,
# sem UDFs nem coleta de registros completos.
#
# POR QUE É IMPORTANTE:
# Promove reutilização, consistência e manutenção
# centralizada das funções do framework.
#
# PERGUNTA RESPONDIDA:
# "Como avaliar desvios estatísticos utilizando
# as mesmas regras em diferentes DataFrames?"


# ============================================================
# FUNÇÃO - REFERÊNCIA SEGURA DE COLUNAS
# ============================================================

# O QUE FAZ:
# Permite referenciar colunas com nomes especiais.
#
# COMO FAZ:
# Escapa crases e utiliza a sintaxe de identificadores
# delimitados do Spark.
#
# POR QUE É IMPORTANTE:
# Evita problemas com nomes contendo pontos ou crases.
#
# PERGUNTA RESPONDIDA:
# "Como acessar colunas independentemente de seus nomes?"

def dq_outlier_col(nome):
    """Retorna uma referência segura para uma coluna Spark."""

    return F.col(
        "`" + nome.replace("`", "``") + "`"
    )


# ============================================================
# FUNÇÃO - VALIDAÇÃO DE VALORES NUMÉRICOS
# ============================================================

# O QUE FAZ:
# Identifica valores numéricos válidos para análise.
#
# COMO FAZ:
# Exclui valores NULL, NaN e infinitos.
#
# POR QUE É IMPORTANTE:
# Evita distorções nos indicadores estatísticos.
#
# PERGUNTA RESPONDIDA:
# "Quais valores podem participar dos cálculos?"

def dq_outlier_valid(coluna):
    """Identifica valores numéricos finitos e não nulos."""

    x = coluna.cast("double")

    return (
        coluna.isNotNull()
        & (~F.isnan(x))
        & (F.abs(x) != F.lit(float("inf")))
    )


# ============================================================
# FUNÇÃO - LIMITES DO INTERVALO INTERQUARTIL
# ============================================================

# O QUE FAZ:
# Calcula os limites inferior e superior do método IQR.
#
# COMO FAZ:
# Utiliza o primeiro e terceiro quartis e um fator
# multiplicador configurável.
#
# POR QUE É IMPORTANTE:
# Permite identificar valores fora da faixa estatística
# definida pelo intervalo interquartil.
#
# PERGUNTA RESPONDIDA:
# "Quais valores estão fora dos limites do IQR?"

def dq_outlier_iqr_bounds(q1, q3, factor=1.5):
    """
    Retorna:
        limite_inferior
        limite_superior
        iqr

    Retorna None para limites inaplicáveis.
    """

    if q1 is None or q3 is None or factor <= 0:
        return None, None, None

    iqr = float(q3) - float(q1)

    if iqr < 0:
        return None, None, None

    limite_inferior = float(q1) - factor * iqr
    limite_superior = float(q3) + factor * iqr

    return limite_inferior, limite_superior, iqr


# ============================================================
# FUNÇÃO - LIMIAR DO MODIFIED Z-SCORE
# ============================================================

# O QUE FAZ:
# Calcula a distância máxima permitida em relação
# à mediana no método Modified Z-Score.
#
# COMO FAZ:
# Utiliza o MAD e o limiar configurado.
#
# POR QUE É IMPORTANTE:
# Permite avaliar valores extremos utilizando uma
# medida robusta de dispersão.
#
# PERGUNTA RESPONDIDA:
# "Qual distância da mediana caracteriza um outlier
# pelo Modified Z-Score?"

def dq_outlier_modified_z_threshold(
    mad,
    threshold=3.5
):
    """
    Retorna o limite absoluto de distância à mediana.

    Retorna None quando o método não é aplicável.
    """

    if mad is None or mad <= 0 or threshold <= 0:
        return None

    return threshold * float(mad) / 0.6745


In [0]:
# ============================================================
# FUNÇÃO - GENÉRICAS DE CORRELATION PROFILE
# ============================================================
#
# O QUE FAZ:
# Cria utilitários reutilizáveis para colunas e classificação.
#
# COMO FAZ:
# Usa funções nativas do Spark e metadados pequenos em Python.
#
# POR QUE É IMPORTANTE:
# Evita UDFs, processamento local de registros e regras de domínio.
#
# PERGUNTA RESPONDIDA:
# "Como manter a análise independente da fonte?"

def dq_corr_col(name):
    return F.col("`" + name.replace("`", "``") + "`")

def dq_corr_valid(name):
    x = dq_corr_col(name).cast("double")
    return x.isNotNull() & (~F.isnan(x)) & (F.abs(x) != F.lit(float("inf")))

def dq_corr_classify(value, weak=0.30, strong=0.80):
    if value is None:
        return "NAO_APLICAVEL", "INDEFINIDA"
    magnitude = abs(float(value))
    intensity = "FORTE" if magnitude >= strong else ("MODERADA" if magnitude >= weak else "FRACA")
    direction = "POSITIVA" if value > 0 else ("NEGATIVA" if value < 0 else "NULA")
    return intensity, direction

def dq_corr_safe_coefficient(value):
    # Corrige apenas ruído numérico muito pequeno além do intervalo teórico.
    if value is None or not (-float("inf") < float(value) < float("inf")):
        return None
    return max(-1.0, min(1.0, float(value)))